### 1. Run Tomofast Inversion

In [ ]:
import os
base_dir = os.getcwd()
tomofast_out = os.path.join(
    base_dir,
    "Tec_files_bellon",
    "converted",
    "Tomofast_Output_inversion"
)
print("Renaming folders in:", tomofast_out)
for name in sorted(os.listdir(tomofast_out)):
    full_path = os.path.join(tomofast_out, name)
    if os.path.isdir(full_path):
        new_name = name
        if new_name.startswith("TEC_"):
            new_name = new_name[len("TEC_"):]
        if new_name.endswith("_inversion"):
            new_name = new_name[:-len("_inversion")]
        if new_name != name:
            new_path = os.path.join(tomofast_out, new_name)
            print(f"Renaming: {name} -> {new_name}")
            os.rename(full_path, new_path)
print("Done.")

### 2. Plot Tomofast Output (Bxyz misfits)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

base_dir = os.getcwd()
tomofast_out = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_inversion")
print("Reading from:", tomofast_out)

# Column count -> {component_name: column_index}
#   3 cols: x y Bz       4 cols: x y z Bz       6+ cols: x y z Bx By Bz
COLUMN_LAYOUTS = {3: {"Bz": 2}, 4: {"Bz": 3}}


def get_components(ncols):
    if ncols in COLUMN_LAYOUTS:
        return COLUMN_LAYOUTS[ncols]
    if ncols >= 6:
        return {"Bx": 3, "By": 4, "Bz": 5}
    return None


def load_2d(path):
    """Load a data file, guaranteeing a 2D array even if it has only one row."""
    data = np.loadtxt(path, skiprows=1)
    return data.reshape(1, -1) if data.ndim == 1 else data


def plot_bxyz(x_coords, y_coords, data, sfx, out_dir, label):
    os.makedirs(out_dir, exist_ok=True)

    components = get_components(data.shape[1])
    if components is None:
        print(f"[SKIP] {sfx} {label}: unsupported number of columns = {data.shape[1]}")
        return

    fig, axes = plt.subplots(1, len(components), figsize=(6 * len(components), 6))
    if len(components) == 1:
        axes = [axes]
    fig.suptitle(f"{sfx} \u2014 {label}", fontsize=14)

    bz_col = data[:, components["Bz"]]
    scale = np.nanmax(np.abs(bz_col))

    for ax, (component, col) in zip(axes, components.items()):
        b_values = -data[:, col] if component == "Bz" else data[:, col]  # flip only Bz, tomofast -> qdm convention
        df = pd.DataFrame({"x": x_coords, "y": y_coords, "b": b_values})
        df = df.sort_values(["y", "x"]).reset_index(drop=True)
        unique_x = np.sort(df["x"].unique())
        unique_y = np.sort(df["y"].unique())
        nx, ny = len(unique_x), len(unique_y)
        if len(df) != nx * ny:
            print(f"[SKIP] {sfx} {label} {component}: not a regular grid")
            continue

        X = df["x"].values.reshape(ny, nx)
        Y = df["y"].values.reshape(ny, nx)
        Z = df["b"].values.reshape(ny, nx)

        im = ax.contourf(X, Y, Z, levels=100, cmap="seismic", vmin=-scale, vmax=scale)
        cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.set_label(f"{component} (nT)", fontsize=12)
        cbar.ax.tick_params(labelsize=11)
        ax.set_xlabel("x ($\\mu$m)", fontsize=12)
        ax.set_ylabel("y ($\\mu$m)", fontsize=12)
        ax.set_title(component, fontsize=13)
        ax.tick_params(labelsize=11)
        ax.set_aspect("equal")
        ax.ticklabel_format(useOffset=False, style="plain")
        ax.xaxis.set_major_locator(plt.MaxNLocator(4))
        ax.yaxis.set_major_locator(plt.MaxNLocator(4))

    plt.tight_layout()
    filename = os.path.join(out_dir, f"Bxyz_{sfx}_{label}.png")
    plt.savefig(filename, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    print("Saved:", filename)


for sfx in sorted(os.listdir(tomofast_out)):
    sample_dir = os.path.join(tomofast_out, sfx)
    if not os.path.isdir(sample_dir):
        continue

    data_dir = os.path.join(sample_dir, "data")
    files_to_plot = {
        "observed": os.path.join(data_dir, "mag_observed.txt"),
        "calculated": os.path.join(data_dir, "mag_final.txt"),
        "misfit": os.path.join(data_dir, "mag_misfit.txt"),
    }

    for label, filepath in files_to_plot.items():
        if not os.path.isfile(filepath):
            print(f"[missing] {filepath}")
            continue

        data = load_2d(filepath)
        x_coords = data[:, 0]
        y_coords = data[:, 1]
        plot_bxyz(x_coords, y_coords, data, sfx, sample_dir, label)

print("\nDone.")

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

base_dir = os.getcwd()
tomofast_out = os.path.join(
    base_dir,
    "Tec_files_bellon",
    "converted",
    "Tomofast_Output_inversion"
)

print("Reading from:", tomofast_out)


def get_components_from_columns(ncols):
    """
    Detect file format based on number of columns.

    Supported formats:
    x y Bz              = 3 columns
    x y z Bz            = 4 columns
    x y z Bx By Bz      = 6 columns
    """

    if ncols == 3:
        return {"Bz": 2}

    elif ncols == 4:
        return {"Bz": 3}

    elif ncols >= 6:
        return {
            "Bx": 3,
            "By": 4,
            "Bz": 5,
        }

    else:
        return None


def plot_component_obs_calc_misfit(sfx, sample_dir):
    os.makedirs(sample_dir, exist_ok=True)

    data_dir = os.path.join(sample_dir, "data")

    files_to_plot = {
        "observed": os.path.join(data_dir, "mag_observed.txt"),
        "calculated": os.path.join(data_dir, "mag_final.txt"),
        "misfit": os.path.join(data_dir, "mag_misfit.txt"),
    }

    data_all = {}

    for label, filepath in files_to_plot.items():
        if not os.path.isfile(filepath):
            print(f"[missing] {filepath}")
            return

        data = np.loadtxt(filepath, skiprows=1)

        if data.ndim == 1:
            data = data.reshape(1, -1)

        data_all[label] = data

    ncols = data_all["observed"].shape[1]
    components = get_components_from_columns(ncols)

    if components is None:
        print(f"[SKIP] {sfx}: unsupported number of columns = {ncols}")
        return

    print(f"{sfx}: detected {ncols} columns -> plotting {list(components.keys())}")

    for component, col in components.items():
        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        fig.suptitle(f"{sfx} - {component}", fontsize=14)

        obs_values = data_all["observed"][:, col]

        # Flip only Bz, tomofast -> qdm convention
        if component == "Bz":
            obs_values = -obs_values

        scale = np.nanmax(np.abs(obs_values))

        for ax, label in zip(axes, ["observed", "calculated", "misfit"]):
            data = data_all[label]

            if data.shape[1] <= col:
                print(f"[SKIP] {sfx} {component} {label}: column {col} missing")
                continue

            x_coords = data[:, 0]
            y_coords = data[:, 1]

            b_values = data[:, col]

            # Flip only Bz, tomofast -> qdm convention
            if component == "Bz":
                b_values = -b_values

            df = pd.DataFrame({
                "x": x_coords,
                "y": y_coords,
                "b": b_values
            })

            df = df.sort_values(["y", "x"]).reset_index(drop=True)

            unique_x = np.sort(df["x"].unique())
            unique_y = np.sort(df["y"].unique())

            nx, ny = len(unique_x), len(unique_y)

            if len(df) != nx * ny:
                print(f"[SKIP] {sfx} {component} {label}: not a regular grid")
                continue

            X = df["x"].values.reshape(ny, nx)
            Y = df["y"].values.reshape(ny, nx)
            Z = df["b"].values.reshape(ny, nx)

            im = ax.contourf(
                X,
                Y,
                Z,
                levels=100,
                cmap="seismic",
                vmin=-scale,
                vmax=scale
            )

            cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
            cbar.set_label(f"{component} (nT)", fontsize=12)
            cbar.ax.tick_params(labelsize=11)

            ax.set_xlabel("x (µm)", fontsize=12)
            ax.set_ylabel("y (µm)", fontsize=12)
            ax.set_title(label, fontsize=13)
            ax.tick_params(labelsize=11)
            ax.set_aspect("equal")
            ax.ticklabel_format(useOffset=False, style="plain")
            ax.xaxis.set_major_locator(plt.MaxNLocator(4))
            ax.yaxis.set_major_locator(plt.MaxNLocator(4))

        plt.tight_layout()

        filename = os.path.join(
            sample_dir,
            f"{component}_obs_calc_misfit_{sfx}.png"
        )

        plt.savefig(filename, dpi=300, bbox_inches="tight")
        plt.show()
        plt.close()

        print("Saved:", filename)


for sfx in sorted(os.listdir(tomofast_out)):
    sample_dir = os.path.join(tomofast_out, sfx)

    if not os.path.isdir(sample_dir):
        continue

    plot_component_obs_calc_misfit(sfx, sample_dir)


print("\nDone.")

### 4- Inversion metrics

In [ ]:
# ------------------------------------------------------------
# Compute inversion fit metrics (data cost, R^2, RMS) per body,
# comparing observed vs. calculated (final model) field data.
# ------------------------------------------------------------
import os
import numpy as np
import pandas as pd

base_dir = os.getcwd()
tomofast_out = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_inversion")
print("Reading from:", tomofast_out)

# Column count -> {component_name: column_index}, covers all 3 file formats:
#   3 cols: x y Bz          4 cols: x y z Bz          6+ cols: x y z Bx By Bz
COLUMN_LAYOUTS = {3: {"Bz": 2}, 4: {"Bz": 3}}


def get_components(ncols):
    if ncols in COLUMN_LAYOUTS:
        return COLUMN_LAYOUTS[ncols]
    if ncols >= 6:
        return {"Bx": 3, "By": 4, "Bz": 5}
    return None


def load_2d(path):
    """Load a data file, guaranteeing a 2D array even if it has only one row."""
    data = np.loadtxt(path, skiprows=1)
    return data.reshape(1, -1) if data.ndim == 1 else data


def calc_metrics(obs, calc):
    """
    Data cost (relative L2 norm) and R^2 between observed and calculated field values.
    NOTE: with these definitions, RMS (root-mean-square residual, normalized by the
    RMS of obs) is mathematically identical to data_cost for any data - the n's
    cancel out (sqrt(mean(r^2))/sqrt(mean(o^2)) == norm(r)/norm(o)). It's returned
    separately only for clarity/readability in the output table, not as an
    independent check. If you want a true (non-normalized) RMS instead, change
    `rms = np.sqrt(np.mean(residuals ** 2))` and drop the division by obs_rms.
    """
    residuals = calc - obs

    data_cost = np.linalg.norm(residuals) / np.linalg.norm(obs)
    #rms = np.sqrt(np.mean(residuals ** 2)) / np.sqrt(np.mean(obs ** 2))
    rms = np.sqrt(np.mean(residuals ** 2))

    ss_res = np.sum(residuals ** 2)
    ss_tot = np.sum((obs - obs.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot != 0 else np.nan

    return data_cost, r2, rms


results = []

for sfx in sorted(os.listdir(tomofast_out)):
    sample_dir = os.path.join(tomofast_out, sfx)
    if not os.path.isdir(sample_dir):
        continue

    obs_file = os.path.join(sample_dir, "data", "mag_observed.txt")
    calc_file = os.path.join(sample_dir, "data", "mag_final.txt")
    if not os.path.isfile(obs_file) or not os.path.isfile(calc_file):
        print(f"[missing obs/calc] {sfx}")
        continue

    obs_data = load_2d(obs_file)
    calc_data = load_2d(calc_file)

    components = get_components(obs_data.shape[1])
    if components is None:
        print(f"[SKIP] {sfx}: unsupported column count = {obs_data.shape[1]}")
        continue

    print(f"{sfx}: detected {obs_data.shape[1]} columns -> calculating {list(components.keys())}")

    for component, col in components.items():
        if calc_data.shape[1] <= col:
            print(f"[SKIP] {sfx} {component}: calculated file missing column {col}")
            continue

        obs_values = obs_data[:, col]
        calc_values = calc_data[:, col]
        data_cost, r2, rms = calc_metrics(obs_values, calc_values)

        results.append({
            "inversion": sfx,
            "component": component,
            "n_data": len(obs_values),
            "data_cost": data_cost,
            "r2": r2,
            "rms": rms,
        })

metrics_df = pd.DataFrame(results)
print("\nMetrics:")
print(metrics_df)

out_csv = os.path.join(tomofast_out, "inversion_metrics.csv")
metrics_df.to_csv(out_csv, index=False)
print("\nSaved:", out_csv)
print("\nDone.")

### 5- Calculate parameters of the inverted models

In [ ]:
# ------------------------------------------------------------
# Per-cell parameters from Tomofast Inverted magnetization + mesh
# LOOP over all body_* folders -> ONE nonzero-only CSV per body
# (Full model: recovered magnetization from the inversion output)
# ------------------------------------------------------------

import os
import glob
import numpy as np
import pandas as pd

base_dir = os.getcwd()
TOMOFAST_ROOT_In = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")
TOMOFAST_ROOT_Out = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_inversion")

UM_TO_M = 1e-6


# ============================================================
# Magnetization -> Dec, Inc, |M|
#
#   |M|   = sqrt(Mx^2 + My^2 + Mz^2)
#   Inc   = arccos(-Mz / |M|) - 90
#           (Mz negated: Tomofast's internal z-axis is positive-down,
#            micromag/QDM convention is positive-up)
#   Dec   = 90 - atan2(My, Mx)
# ============================================================
def dec_inc_norm(Mx, My, Mz):
    Mx, My, Mz = np.atleast_1d(Mx, My, Mz)
    norm = np.sqrt(Mx**2 + My**2 + Mz**2)
    Mz_used = -Mz
    safe_norm = np.where(norm > 0, norm, np.nan)
    theta = np.degrees(np.arccos(np.clip(Mz_used / safe_norm, -1, 1)))
    phi = np.degrees(np.arctan2(My, Mx))
    inc = theta - 90.0
    dec = np.mod(90.0 - phi, 360.0)
    return dec, inc, norm


# ============================================================
# Fisher (1953) 95% confidence cone around the mean direction
#
#   alpha95 = arccos[ 1 - ((N-R)/R) * ((1/0.05)^(1/(N-1)) - 1) ]
#
# N = number of unit vectors, R = resultant length of their sum
# ============================================================
def alpha95(N, R):
    if R <= 0 or N <= 1:
        return np.nan
    term = (1 / 0.05) ** (1 / (N - 1)) - 1
    val = 1 - ((N - R) / R) * term
    val = np.clip(val, -1, 1)  # guard against tiny floating-point overshoot
    return np.degrees(np.arccos(val))


body_dirs = sorted(
    d for d in os.listdir(TOMOFAST_ROOT_In)
    if d.startswith("body_") and os.path.isdir(os.path.join(TOMOFAST_ROOT_In, d))
)
print("Found bodies:", body_dirs)

for body in body_dirs:
    print("\n===================================================")
    print("Processing", body)
    print("===================================================")

    body_folder_Out = os.path.join(TOMOFAST_ROOT_Out, body)
    body_folder_In = os.path.join(TOMOFAST_ROOT_In, body)
    mag_matches = glob.glob(os.path.join(body_folder_Out, "model", "mag_final_model_full.txt"))
    mesh_matches = glob.glob(os.path.join(body_folder_In, "Meshgrid_*.txt"))

    if not mag_matches or not mesh_matches:
        print("   Missing Magnetization or Meshgrid file - skipping.")
        continue

    mesh = pd.read_csv(mesh_matches[0], sep=r"\s+", skiprows=1, header=None,
                        names=["X1", "X2", "Y1", "Y2", "Z1", "Z2", "i", "j", "k"])
    mag = pd.read_csv(mag_matches[0], sep=r"\s+", skiprows=1, header=None,
                       names=["Mx", "My", "Mz"])

    if len(mesh) != len(mag):
        print(f"   Cell count mismatch ({len(mesh)} vs {len(mag)}) - skipping.")
        continue

    df = pd.concat([mesh, mag], axis=1)
    df.insert(0, "CellID", np.arange(1, len(df) + 1))
    print(f"   Loaded {len(df)} cells.")

    # ------------------------------------------------------------
    # Cell geometry and volume
    #   dx = X2-X1, dy = Y2-Y1, dz = Z2-Z1   (converted um -> m)
    #   V  = dx * dy * dz
    # ------------------------------------------------------------
    df["dx"] = (df["X2"] - df["X1"]) * UM_TO_M
    df["dy"] = (df["Y2"] - df["Y1"]) * UM_TO_M
    df["dz"] = (df["Z2"] - df["Z1"]) * UM_TO_M
    df["Volume"] = df["dx"] * df["dy"] * df["dz"]

    # ------------------------------------------------------------
    # Per-cell direction and magnitude (see dec_inc_norm above),
    # from the INVERTED (recovered) magnetization
    # ------------------------------------------------------------
    df["Dec"], df["Inc"], df["Mag_norm"] = dec_inc_norm(df["Mx"], df["My"], df["Mz"])

    # ------------------------------------------------------------
    # Per-cell recovered magnetic moment
    #   m = M * V        (scalar, A.m^2)
    #   m_vec = (Mx,My,Mz) * V   (vector components, A.m^2)
    # ------------------------------------------------------------
    df["Moment"] = df["Mag_norm"] * df["Volume"]
    df["mx"] = df["Mx"] * df["Volume"]
    df["my"] = df["My"] * df["Volume"]
    df["mz"] = df["Mz"] * df["Volume"]

    cols = ["CellID", "i", "j", "k", "X1", "X2", "Y1", "Y2", "Z1", "Z2",
            "dx", "dy", "dz", "Volume", "Mx", "My", "Mz",
            "mx", "my", "mz",
            "Mag_norm", "Dec", "Inc", "Moment"]
    nz_out = df.loc[df["Mag_norm"] > 0, cols].copy()

    n = len(nz_out)
    if n == 0:
        print("   No nonzero cells found - skipping save.")
        continue

    # ============================================================
    # Fisher mean direction (Fisher, 1953)
    #
    #   unit vector per cell:  u_i = (Mx,My,Mz)_i / |M_i|
    #   resultant vector:      (X,Y,Z) = sum(u_i)
    #   resultant length:      R = sqrt(X^2 + Y^2 + Z^2)   [0 <= R <= N]
    #   mean direction:        Dec/Inc of (X,Y,Z)
    #
    # Equal weight per cell, independent of cell magnitude - avoids the
    # few-strong-cells bias that a plain component average would have.
    # ============================================================
    norm_per_cell = np.sqrt(nz_out["Mx"]**2 + nz_out["My"]**2 + nz_out["Mz"]**2)
    ux = nz_out["Mx"] / norm_per_cell
    uy = nz_out["My"] / norm_per_cell
    uz = nz_out["Mz"] / norm_per_cell

    X, Y, Z = ux.sum(), uy.sum(), uz.sum()
    R = np.sqrt(X**2 + Y**2 + Z**2)
    fisher_dec, fisher_inc, _ = dec_inc_norm(X, Y, Z)
    fisher_alpha95 = alpha95(n, R)

    # ------------------------------------------------------------
    # Amplitude-weighted mean direction (for comparison with Fisher mean)
    #   Dec/Inc of the simple component average (Mx_mean, My_mean, Mz_mean)
    # ------------------------------------------------------------
    mean_dec, mean_inc, _ = dec_inc_norm(nz_out["Mx"].mean(), nz_out["My"].mean(), nz_out["Mz"].mean())

    # ============================================================
    # Recovered (bulk) moment - vector sum over all nonzero cells
    #
    #   m_recovered = sum(m_i)                  (components summed first)
    #   |m_recovered| = sqrt(mx^2 + my^2 + mz^2)
    #
    # Preserves directional cancellation between cells - this is the
    # physically meaningful bulk moment the inversion recovered.
    # ============================================================
    recovered_moment = np.array([
        nz_out["mx"].sum(),
        nz_out["my"].sum(),
        nz_out["mz"].sum()
    ])

    summary = {
        "Mean_Mag":                    nz_out["Mag_norm"].mean(),
        "Mean_Dec":                    mean_dec[0],
        "Mean_Inc":                    mean_inc[0],
        "Fisher_Dec":                  fisher_dec[0],
        "Fisher_Inc":                  fisher_inc[0],
        "Fisher_R":                    R,
        "Fisher_R_over_N":             R / n,
        "Fisher_alpha95":              fisher_alpha95,
        "Mean_Moment":                 nz_out["Moment"].mean(),
        "Recovered_mx":                recovered_moment[0],
        "Recovered_my":                recovered_moment[1],
        "Recovered_mz":                recovered_moment[2],
        "Recovered_moment_magnitude":  np.linalg.norm(recovered_moment),
    }

    print(f"   N_nonzero_cells = {n}")
    for key, val in summary.items():
        print(f"   {key:22s}: {val:,.25f}")

    for key in summary:
        nz_out[key] = np.nan
    nz_out.loc[nz_out.index[0], list(summary.keys())] = list(summary.values())

    out_path = os.path.join(body_folder_Out, f"{body}_nonzero.csv")
    nz_out.to_csv(out_path, index=False)
    print("   Saved:", out_path)

print("\nDONE.")

In [ ]:
# ------------------------------------------------------------
# Per-cell parameters from Tomofast Inverted magnetization + mesh
# LOOP over all body_* folders -> ONE CSV per body PER threshold
# (Thresholded model: keep only cells with |M| >= fraction * max|M|)
# ------------------------------------------------------------

import os
import glob
import numpy as np
import pandas as pd

base_dir = os.getcwd()
TOMOFAST_ROOT_In  = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_forward")
TOMOFAST_ROOT_Out = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_inversion")

UM_TO_M = 1e-6

# ------------------------------------------------------------
# THRESHOLDS: fraction of max |M| per body
# 0.0 = all nonzero cells, 0.5 = top 50%, 0.9 = top 10% etc.
# ------------------------------------------------------------
THRESHOLDS = np.arange(0.0, 1.0, 0.1)   # 0.0, 0.1, 0.2 ... 0.9


# ============================================================
# Magnetization -> Dec, Inc, |M|
#
#   |M|   = sqrt(Mx^2 + My^2 + Mz^2)
#   Inc   = arccos(-Mz / |M|) - 90
#           (Mz negated: Tomofast's internal z-axis is positive-down,
#            micromag/QDM convention is positive-up)
#   Dec   = 90 - atan2(My, Mx)
# ============================================================
def dec_inc_norm(Mx, My, Mz):
    Mx, My, Mz = np.atleast_1d(Mx, My, Mz)
    norm = np.sqrt(Mx**2 + My**2 + Mz**2)
    Mz_used = -Mz
    safe_norm = np.where(norm > 0, norm, np.nan)
    theta = np.degrees(np.arccos(np.clip(Mz_used / safe_norm, -1, 1)))
    phi = np.degrees(np.arctan2(My, Mx))
    inc = theta - 90.0
    dec = np.mod(90.0 - phi, 360.0)
    return dec, inc, norm


# ============================================================
# Fisher (1953) 95% confidence cone around the mean direction
#
#   alpha95 = arccos[ 1 - ((N-R)/R) * ((1/0.05)^(1/(N-1)) - 1) ]
#
# N = number of unit vectors, R = resultant length of their sum
# ============================================================
def alpha95(N, R):
    if R <= 0 or N <= 1:
        return np.nan
    term = (1 / 0.05) ** (1 / (N - 1)) - 1
    val = 1 - ((N - R) / R) * term
    val = np.clip(val, -1, 1)  # guard against tiny floating-point overshoot
    return np.degrees(np.arccos(val))


body_dirs = sorted(
    d for d in os.listdir(TOMOFAST_ROOT_In)
    if d.startswith("body_") and os.path.isdir(os.path.join(TOMOFAST_ROOT_In, d))
)
print("Found bodies:", body_dirs)

for body in body_dirs:
    print("\n===================================================")
    print("Processing", body)
    print("===================================================")

    body_folder_Out = os.path.join(TOMOFAST_ROOT_Out, body)
    body_folder_In  = os.path.join(TOMOFAST_ROOT_In,  body)
    mag_matches  = glob.glob(os.path.join(body_folder_Out, "model", "mag_final_model_full.txt"))
    mesh_matches = glob.glob(os.path.join(body_folder_In,  "Meshgrid_*.txt"))

    if not mag_matches or not mesh_matches:
        print("   Missing Magnetization or Meshgrid file - skipping.")
        continue

    mesh = pd.read_csv(mesh_matches[0], sep=r"\s+", skiprows=1, header=None,
                        names=["X1", "X2", "Y1", "Y2", "Z1", "Z2", "i", "j", "k"])
    mag  = pd.read_csv(mag_matches[0],  sep=r"\s+", skiprows=1, header=None,
                        names=["Mx", "My", "Mz"])

    if len(mesh) != len(mag):
        print(f"   Cell count mismatch ({len(mesh)} vs {len(mag)}) - skipping.")
        continue

    df = pd.concat([mesh, mag], axis=1)
    df.insert(0, "CellID", np.arange(1, len(df) + 1))
    print(f"   Loaded {len(df)} cells.")

    # ------------------------------------------------------------
    # Cell geometry and volume
    #   dx = X2-X1, dy = Y2-Y1, dz = Z2-Z1   (converted um -> m)
    #   V  = dx * dy * dz
    # ------------------------------------------------------------
    df["dx"]     = (df["X2"] - df["X1"]) * UM_TO_M
    df["dy"]     = (df["Y2"] - df["Y1"]) * UM_TO_M
    df["dz"]     = (df["Z2"] - df["Z1"]) * UM_TO_M
    df["Volume"] = df["dx"] * df["dy"] * df["dz"]

    # ------------------------------------------------------------
    # Per-cell direction, magnitude, and moment (see formulas above)
    #   Moment = |M| * V
    #   m_vec  = (Mx,My,Mz) * V
    # ------------------------------------------------------------
    df["Dec"], df["Inc"], df["Mag_norm"] = dec_inc_norm(df["Mx"], df["My"], df["Mz"])
    df["Moment"] = df["Mag_norm"] * df["Volume"]
    df["mx"] = df["Mx"] * df["Volume"]
    df["my"] = df["My"] * df["Volume"]
    df["mz"] = df["Mz"] * df["Volume"]

    cols = ["CellID", "i", "j", "k", "X1", "X2", "Y1", "Y2", "Z1", "Z2",
            "dx", "dy", "dz", "Volume", "Mx", "My", "Mz",
            "mx", "my", "mz",
            "Mag_norm", "Dec", "Inc", "Moment"]

    # all cells (base, unfiltered) - thresholding is applied per-loop below
    nz_all = df[cols].copy()
    mag_max = nz_all["Mag_norm"].max()

    if len(nz_all) == 0:
        print("   No nonzero cells - skipping.")
        continue

    print(f"   Mag_norm max = {mag_max:.6f}")

    # ------------------------------------------------------------
    # LOOP over thresholds
    #   keep cells where |M| >= thresh_fraction * max(|M|)
    # ------------------------------------------------------------
    for thresh in THRESHOLDS:
        thresh_abs = thresh * mag_max   # absolute cutoff
        nz_out = nz_all.loc[nz_all["Mag_norm"] >= thresh_abs].copy()

        n = len(nz_out)
        if n == 0:
            print(f"   threshold={thresh:.1f}: no cells left - skipping.")
            continue

        # ============================================================
        # Fisher mean direction (Fisher, 1953)
        #
        #   unit vector per cell:  u_i = (Mx,My,Mz)_i / |M_i|
        #   resultant vector:      (X,Y,Z) = sum(u_i)
        #   resultant length:      R = sqrt(X^2 + Y^2 + Z^2)   [0 <= R <= N]
        #   mean direction:        Dec/Inc of (X,Y,Z)
        #
        # Equal weight per cell, independent of cell magnitude - avoids the
        # few-strong-cells bias that a plain component average would have.
        # ============================================================
        norm_per_cell = np.sqrt(nz_out["Mx"]**2 + nz_out["My"]**2 + nz_out["Mz"]**2)
        ux = nz_out["Mx"] / norm_per_cell
        uy = nz_out["My"] / norm_per_cell
        uz = nz_out["Mz"] / norm_per_cell

        X, Y, Z = ux.sum(), uy.sum(), uz.sum()
        R = np.sqrt(X**2 + Y**2 + Z**2)
        fisher_dec, fisher_inc, _ = dec_inc_norm(X, Y, Z)
        fisher_alpha95 = alpha95(n, R)

        # ------------------------------------------------------------
        # Amplitude-weighted mean direction (for comparison with Fisher mean)
        # ------------------------------------------------------------
        mean_dec, mean_inc, _ = dec_inc_norm(
            nz_out["Mx"].mean(),
            nz_out["My"].mean(),
            nz_out["Mz"].mean()
        )

        # ============================================================
        # Recovered (bulk) moment - vector sum over all cells at this
        # threshold
        #
        #   m_recovered = sum(m_i)
        #   |m_recovered| = sqrt(mx^2 + my^2 + mz^2)
        # ============================================================
        recovered_moment = np.array([
            nz_out["mx"].sum(),
            nz_out["my"].sum(),
            nz_out["mz"].sum()
        ])

        summary = {
            "Threshold_fraction":          thresh,
            "Threshold_abs":               thresh_abs,
            "N_cells":                     n,
            "Mean_Mag":                    nz_out["Mag_norm"].mean(),
            "Mean_Dec":                    mean_dec[0],
            "Mean_Inc":                    mean_inc[0],
            "Fisher_Dec":                  fisher_dec[0],
            "Fisher_Inc":                  fisher_inc[0],
            "Fisher_R":                    R,
            "Fisher_R_over_N":             R / n,
            "Fisher_alpha95":              fisher_alpha95,
            "Mean_Moment":                 nz_out["Moment"].mean(),
            "Recovered_mx":                recovered_moment[0],
            "Recovered_my":                recovered_moment[1],
            "Recovered_mz":                recovered_moment[2],
            "Recovered_moment_magnitude":  np.linalg.norm(recovered_moment),
        }

        print(f"   threshold={thresh:.1f} | N_cells={n} | "
              f"R/N={R/n:.3f} | alpha95={fisher_alpha95:.2f} | "
              f"Recovered_mag={np.linalg.norm(recovered_moment):.4e}")

        # write summary to first row only
        for key in summary:
            nz_out[key] = np.nan
        nz_out.loc[nz_out.index[0], list(summary.keys())] = list(summary.values())

        # save one CSV per threshold
        thresh_str = f"{thresh:.1f}".replace(".", "p")   # 0.1 -> 0p1
        out_path = os.path.join(
            body_folder_Out,
            f"{body}_nonzero_thresh{thresh_str}.csv"
        )
        nz_out.to_csv(out_path, index=False)
        print(f"   Saved: {out_path}")

print("\nDONE.")

### analysis the model

In [ ]:
# ============================================================
# STANDALONE: find best-matching threshold (shape/size recovery)
# per body, with a verification plot showing the metric curves
# and the chosen best threshold marked
# ============================================================
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
base_dir = os.getcwd()
TOMOFAST_ROOT_In  = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")
TOMOFAST_ROOT_Out = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_inversion")

THRESHOLDS = np.arange(0.0, 1.0, 0.01)   # 0.0, 0.1, 0.2 ... 0.9

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
})

body_dirs = sorted(
    d for d in os.listdir(TOMOFAST_ROOT_In)
    if d.startswith("body_") and os.path.isdir(os.path.join(TOMOFAST_ROOT_In, d))
)
print("Found bodies:", body_dirs)

best_match_results = []

for body in body_dirs:
    print("\n===================================================")
    print("Processing", body)
    print("===================================================")

    body_folder_Out = os.path.join(TOMOFAST_ROOT_Out, body)
    body_folder_In  = os.path.join(TOMOFAST_ROOT_In,  body)
    mag_matches      = glob.glob(os.path.join(body_folder_Out, "model", "mag_final_model_full.txt"))
    mesh_matches     = glob.glob(os.path.join(body_folder_In,  "Meshgrid_*.txt"))
    true_mag_matches = glob.glob(os.path.join(body_folder_In,  "Magnetization_*.txt"))

    if not mag_matches or not mesh_matches or not true_mag_matches:
        print("   Missing Magnetization, Meshgrid, or True magnetization file - skipping.")
        continue

    mesh = pd.read_csv(mesh_matches[0], sep=r"\s+", skiprows=1, header=None,
                        names=["X1", "X2", "Y1", "Y2", "Z1", "Z2", "i", "j", "k"])
    mag = pd.read_csv(mag_matches[0], sep=r"\s+", skiprows=1, header=None,
                       names=["Mx", "My", "Mz"])
    true_mag = pd.read_csv(true_mag_matches[0], sep=r"\s+", skiprows=1, header=None,
                            names=["Mx", "My", "Mz"])

    if len(mesh) != len(mag) or len(mesh) != len(true_mag):
        print(f"   Cell count mismatch (mesh={len(mesh)}, mag={len(mag)}, true_mag={len(true_mag)}) - skipping.")
        continue

    print(f"   Loaded {len(mesh)} cells.")

    # ------------------------------------------------------------
    # Build 3D grids (inverted model + true model), same i,j,k grid
    # ------------------------------------------------------------
    nx, ny, nz = mesh["i"].max(), mesh["j"].max(), mesh["k"].max()

    Mag_norm = np.sqrt(mag["Mx"]**2 + mag["My"]**2 + mag["Mz"]**2)
    model = np.zeros((nx, ny, nz))
    model[mesh["i"] - 1, mesh["j"] - 1, mesh["k"] - 1] = Mag_norm
    mag_max = Mag_norm.max()

    True_Mag_norm = np.sqrt(true_mag["Mx"]**2 + true_mag["My"]**2 + true_mag["Mz"]**2)
    true_model = np.zeros((nx, ny, nz))
    true_model[mesh["i"] - 1, mesh["j"] - 1, mesh["k"] - 1] = True_Mag_norm
    true_filled = (true_model > 0)

    if not true_filled.any():
        print("   No nonzero true cells - skipping.")
        continue

    # ------------------------------------------------------------
    # Cell volume (assumes uniform cell size, from first cell)
    # ------------------------------------------------------------
    cell_vol = ((mesh["X2"] - mesh["X1"]).iloc[0] *
                (mesh["Y2"] - mesh["Y1"]).iloc[0] *
                (mesh["Z2"] - mesh["Z1"]).iloc[0])
    true_volume = true_filled.sum() * cell_vol

    # ------------------------------------------------------------
    # Threshold sweep: Dice, Jaccard, Precision, Recall, Volume
    # ------------------------------------------------------------
    body_sweep = []
    for thresh in THRESHOLDS:
        thresh_abs = thresh * mag_max
        recovered_filled = (abs(model) >= thresh_abs)

        TP = np.sum(true_filled & recovered_filled)
        FP = np.sum(~true_filled & recovered_filled)
        FN = np.sum(true_filled & ~recovered_filled)

        dice      = 2*TP / (2*TP+FP+FN) if (2*TP+FP+FN) > 0 else np.nan
        jaccard   = TP / (TP+FP+FN) if (TP+FP+FN) > 0 else np.nan
        precision = TP / (TP+FP) if (TP+FP) > 0 else np.nan
        recall    = TP / (TP+FN) if (TP+FN) > 0 else np.nan
        recovered_volume = recovered_filled.sum() * cell_vol

        body_sweep.append(dict(
            threshold=thresh, Dice=dice, Jaccard=jaccard,
            Precision=precision, Recall=recall,
            Recovered_volume=recovered_volume,
        ))

    body_sweep_df = pd.DataFrame(body_sweep)

    if body_sweep_df["Dice"].isna().all():
        print("   No valid Dice scores at any threshold - skipping.")
        continue

    # ------------------------------------------------------------
    # Pick the threshold with the highest Dice -> "best recovered grain"
    # ------------------------------------------------------------
    best_row = body_sweep_df.loc[body_sweep_df["Dice"].idxmax()]

    best_match_results.append({
        "body":              body,
        "Best_threshold":    best_row["threshold"],
        "Dice":              best_row["Dice"],
        "Jaccard":           best_row["Jaccard"],
        "Precision":         best_row["Precision"],
        "Recall":            best_row["Recall"],
        "True_volume":       true_volume,
        "Recovered_volume":  best_row["Recovered_volume"],
        "Volume_ratio":      best_row["Recovered_volume"] / true_volume,
    })

    print(f"   Best threshold = {best_row['threshold']:.4f} | "
          f"Dice={best_row['Dice']:.3f} | Jaccard={best_row['Jaccard']:.3f} | "
          f"Precision={best_row['Precision']:.3f} | Recall={best_row['Recall']:.3f} | "
          f"Volume ratio={best_row['Recovered_volume']/true_volume:.2f}")

    # ============================================================
    # VERIFICATION PLOT - metric curves vs threshold, best point marked
    # ============================================================
    fig, ax = plt.subplots(figsize=(7, 5))

    ax.plot(body_sweep_df["threshold"], body_sweep_df["Dice"], "o-", label="Dice", color="#1f77b4")
    ax.plot(body_sweep_df["threshold"], body_sweep_df["Jaccard"], "s-", label="Jaccard", color="#ff7f0e")
    ax.plot(body_sweep_df["threshold"], body_sweep_df["Precision"], "^-", label="Precision", color="#2ca02c")
    ax.plot(body_sweep_df["threshold"], body_sweep_df["Recall"], "v-", label="Recall", color="#d62728")

    # mark the chosen best threshold
    ax.axvline(best_row["threshold"], color="gray", linestyle="--", linewidth=1.2)
    ax.scatter([best_row["threshold"]], [best_row["Dice"]], color="black", zorder=5, s=80,
               label=f"Best (Dice={best_row['Dice']:.2f})")

    ax.set_xlabel("Threshold (fraction of max |M|)")
    ax.set_ylabel("Score (0-1)")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"{body} - shape match vs threshold")
    ax.legend(fontsize=9, loc="best")
    ax.grid(True, alpha=0.4)

    fig.tight_layout()
    fig_path = os.path.join(body_folder_Out, f"{body}_shape_match_curve.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"   Saved plot: {fig_path}")

# ------------------------------------------------------------
# Save final summary across all bodies
# ------------------------------------------------------------
best_match_df = pd.DataFrame(best_match_results)
print("\n", best_match_df.to_string(index=False))

out_path = os.path.join(TOMOFAST_ROOT_Out, "best_threshold_shape_match.csv")
best_match_df.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("\nDONE.")

In [ ]:
# ============================================================
# STANDALONE: position-independent shape match (re-centered Dice)
# Separates "wrong shape" from "right shape, shifted location"
# ============================================================
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
base_dir = os.getcwd()
TOMOFAST_ROOT_In  = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")
TOMOFAST_ROOT_Out = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_inversion")

THRESHOLDS = np.arange(0.0, 1.0, 0.01)

plt.rcParams.update({
    "font.family": "serif",
    "font.size": 12,
    "axes.labelsize": 12,
    "axes.titlesize": 13,
})


def shift_to_centroid(mask, target_centroid):
    """Shift a binary 3D mask so its centroid aligns with target_centroid."""
    ii, jj, kk = np.where(mask)
    if len(ii) == 0:
        return mask
    current_centroid = np.array([ii.mean(), jj.mean(), kk.mean()])
    shift = np.round(target_centroid - current_centroid).astype(int)

    shifted = np.zeros_like(mask)
    new_ii = ii + shift[0]
    new_jj = jj + shift[1]
    new_kk = kk + shift[2]

    valid = ((new_ii >= 0) & (new_ii < mask.shape[0]) &
             (new_jj >= 0) & (new_jj < mask.shape[1]) &
             (new_kk >= 0) & (new_kk < mask.shape[2]))
    shifted[new_ii[valid], new_jj[valid], new_kk[valid]] = True
    return shifted


def shape_metrics(true_filled, recovered_filled):
    """Standard, position-sensitive metrics."""
    TP = np.sum(true_filled & recovered_filled)
    FP = np.sum(~true_filled & recovered_filled)
    FN = np.sum(true_filled & ~recovered_filled)
    dice      = 2*TP / (2*TP+FP+FN) if (2*TP+FP+FN) > 0 else np.nan
    jaccard   = TP / (TP+FP+FN) if (TP+FP+FN) > 0 else np.nan
    precision = TP / (TP+FP) if (TP+FP) > 0 else np.nan
    recall    = TP / (TP+FN) if (TP+FN) > 0 else np.nan
    return dict(Dice=dice, Jaccard=jaccard, Precision=precision, Recall=recall)


def shape_metrics_position_independent(true_filled, recovered_filled, true_centroid):
    """Dice/Jaccard/Precision/Recall AFTER re-centering recovered onto true's centroid."""
    recovered_recentered = shift_to_centroid(recovered_filled, true_centroid)
    TP = np.sum(true_filled & recovered_recentered)
    FP = np.sum(~true_filled & recovered_recentered)
    FN = np.sum(true_filled & ~recovered_recentered)
    dice      = 2*TP / (2*TP+FP+FN) if (2*TP+FP+FN) > 0 else np.nan
    jaccard   = TP / (TP+FP+FN) if (TP+FP+FN) > 0 else np.nan
    precision = TP / (TP+FP) if (TP+FP) > 0 else np.nan
    recall    = TP / (TP+FN) if (TP+FN) > 0 else np.nan
    return dict(Dice=dice, Jaccard=jaccard, Precision=precision, Recall=recall)


body_dirs = sorted(
    d for d in os.listdir(TOMOFAST_ROOT_In)
    if d.startswith("body_") and os.path.isdir(os.path.join(TOMOFAST_ROOT_In, d))
)
print("Found bodies:", body_dirs)

best_match_results = []

for body in body_dirs:
    print("\n===================================================")
    print("Processing", body)
    print("===================================================")

    body_folder_Out = os.path.join(TOMOFAST_ROOT_Out, body)
    body_folder_In  = os.path.join(TOMOFAST_ROOT_In,  body)
    mag_matches      = glob.glob(os.path.join(body_folder_Out, "model", "mag_final_model_full.txt"))
    mesh_matches     = glob.glob(os.path.join(body_folder_In,  "Meshgrid_*.txt"))
    true_mag_matches = glob.glob(os.path.join(body_folder_In,  "Magnetization_*.txt"))

    if not mag_matches or not mesh_matches or not true_mag_matches:
        print("   Missing Magnetization, Meshgrid, or True magnetization file - skipping.")
        continue

    mesh = pd.read_csv(mesh_matches[0], sep=r"\s+", skiprows=1, header=None,
                        names=["X1", "X2", "Y1", "Y2", "Z1", "Z2", "i", "j", "k"])
    mag = pd.read_csv(mag_matches[0], sep=r"\s+", skiprows=1, header=None,
                       names=["Mx", "My", "Mz"])
    true_mag = pd.read_csv(true_mag_matches[0], sep=r"\s+", skiprows=1, header=None,
                            names=["Mx", "My", "Mz"])

    if len(mesh) != len(mag) or len(mesh) != len(true_mag):
        print(f"   Cell count mismatch (mesh={len(mesh)}, mag={len(mag)}, true_mag={len(true_mag)}) - skipping.")
        continue

    print(f"   Loaded {len(mesh)} cells.")

    nx, ny, nz = mesh["i"].max(), mesh["j"].max(), mesh["k"].max()

    Mag_norm = np.sqrt(mag["Mx"]**2 + mag["My"]**2 + mag["Mz"]**2)
    model = np.zeros((nx, ny, nz))
    model[mesh["i"] - 1, mesh["j"] - 1, mesh["k"] - 1] = Mag_norm
    mag_max = Mag_norm.max()

    True_Mag_norm = np.sqrt(true_mag["Mx"]**2 + true_mag["My"]**2 + true_mag["Mz"]**2)
    true_model = np.zeros((nx, ny, nz))
    true_model[mesh["i"] - 1, mesh["j"] - 1, mesh["k"] - 1] = True_Mag_norm
    true_filled = (true_model > 0)

    if not true_filled.any():
        print("   No nonzero true cells - skipping.")
        continue

    ti, tj, tk = np.where(true_filled)
    true_centroid = np.array([ti.mean(), tj.mean(), tk.mean()])

    cell_vol = ((mesh["X2"] - mesh["X1"]).iloc[0] *
                (mesh["Y2"] - mesh["Y1"]).iloc[0] *
                (mesh["Z2"] - mesh["Z1"]).iloc[0])
    true_volume = true_filled.sum() * cell_vol

    # ------------------------------------------------------------
    # Threshold sweep: BOTH metrics side by side
    # ------------------------------------------------------------
    body_sweep = []
    for thresh in THRESHOLDS:
        thresh_abs = thresh * mag_max
        recovered_filled = (abs(model) >= thresh_abs)

        m_pos    = shape_metrics(true_filled, recovered_filled)
        m_recntr = shape_metrics_position_independent(true_filled, recovered_filled, true_centroid)
        recovered_volume = recovered_filled.sum() * cell_vol

        # estimated shift distance (in cells) between true and recovered centroids
        ri, rj, rk = np.where(recovered_filled)
        if len(ri) > 0:
            rec_centroid = np.array([ri.mean(), rj.mean(), rk.mean()])
            shift_dist = np.linalg.norm(rec_centroid - true_centroid)
        else:
            shift_dist = np.nan

        body_sweep.append(dict(
            threshold=thresh,
            Dice=m_pos["Dice"], Jaccard=m_pos["Jaccard"],
            Precision=m_pos["Precision"], Recall=m_pos["Recall"],
            Dice_recentered=m_recntr["Dice"], Jaccard_recentered=m_recntr["Jaccard"],
            Precision_recentered=m_recntr["Precision"], Recall_recentered=m_recntr["Recall"],
            Shift_distance_cells=shift_dist,
            Recovered_volume=recovered_volume,
        ))

    body_sweep_df = pd.DataFrame(body_sweep)

    if body_sweep_df["Dice"].isna().all():
        print("   No valid Dice scores at any threshold - skipping.")
        continue

    # Best threshold picked using the ORIGINAL (position-sensitive) Dice
    best_row = body_sweep_df.loc[body_sweep_df["Dice"].idxmax()]

    best_match_results.append({
        "body":                  body,
        "Best_threshold":        best_row["threshold"],
        "Dice":                  best_row["Dice"],
        "Dice_recentered":       best_row["Dice_recentered"],
        "Dice_gain_from_shift":  best_row["Dice_recentered"] - best_row["Dice"],
        "Shift_distance_cells":  best_row["Shift_distance_cells"],
        "Precision":             best_row["Precision"],
        "Recall":                best_row["Recall"],
        "True_volume":           true_volume,
        "Recovered_volume":      best_row["Recovered_volume"],
        "Volume_ratio":          best_row["Recovered_volume"] / true_volume,
    })

    print(f"   Best threshold = {best_row['threshold']:.3f}")
    print(f"   Dice (position-sensitive) = {best_row['Dice']:.3f}")
    print(f"   Dice (re-centered)        = {best_row['Dice_recentered']:.3f}")
    print(f"   Shift distance (cells)    = {best_row['Shift_distance_cells']:.2f}")
    print(f"   --> Improvement from re-centering = {best_row['Dice_recentered']-best_row['Dice']:+.3f}")

    # ============================================================
    # VERIFICATION PLOT - both Dice curves overlaid
    # ============================================================
    fig, ax = plt.subplots(figsize=(7, 5))
    ax.plot(body_sweep_df["threshold"], body_sweep_df["Dice"], "-",
            label="Dice (position-sensitive)", color="#1f77b4")
    ax.plot(body_sweep_df["threshold"], body_sweep_df["Dice_recentered"], "-",
            label="Dice (re-centered)", color="#d62728")
    ax.axvline(best_row["threshold"], color="gray", linestyle="--", linewidth=1.0)
    ax.set_xlabel("Threshold (fraction of max |M|)")
    ax.set_ylabel("Dice score (0-1)")
    ax.set_ylim(-0.02, 1.02)
    ax.set_title(f"{body} - shape match: position-sensitive vs re-centered")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.4)
    fig.tight_layout()
    fig_path = os.path.join(body_folder_Out, f"{body}_dice_recentered_comparison.png")
    fig.savefig(fig_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"   Saved plot: {fig_path}")

# ------------------------------------------------------------
# Save final summary across all bodies
# ------------------------------------------------------------
best_match_df = pd.DataFrame(best_match_results)
print("\n", best_match_df.to_string(index=False))

out_path = os.path.join(TOMOFAST_ROOT_Out, "best_threshold_shape_match_recentered.csv")
best_match_df.to_csv(out_path, index=False)
print("\nSaved:", out_path)
print("\nDONE.")

### 6- 3D plot voxels

### 7- recovered misfits plots

In [ ]:
#against thresholds
%matplotlib inline
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import cm

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
base_dir     = os.getcwd()
FORWARD_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")
INVERSE_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_inversion")
METRICS_CSV  = os.path.join(INVERSE_ROOT, "inversion_metrics.csv")
GRAIN_CSV    = r"C:\Users\23121159\OneDrive - Curtin\Ongoing\JGR_Review\Work\1-Merrill\Particle_morphology_data\Grain_size_distirbution.csv"

THRESHOLDS = np.arange(0.0, 1.0, 0.1)   # must match what was used to generate CSVs

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      13,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "pdf.fonttype":   42,
    "ps.fonttype":    42,
})

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def extract_grain_id(body_name):
    return body_name.split("_")[1]


def emax_bin(val):
    for i in range(len(emax_bins) - 1):
        if emax_bins[i] <= val < emax_bins[i + 1]:
            return i
    return len(emax_bins) - 2


def r2_to_size(r2):
    if np.isnan(r2):
        return 60
    if r2 > 0.95:
        return 120
    elif r2 > 0.85:
        return 60
    else:
        return 20


def vec_to_dec_inc(Mx, My, Mz):
    """
    Vector -> (Dec, Inc) in degrees, using the same convention as the rest
    of the pipeline (Mz flipped internally, Inc = theta-90, Dec = 90-phi).
    """
    norm = np.sqrt(Mx ** 2 + My ** 2 + Mz ** 2)
    theta = np.degrees(np.arccos(np.clip(-Mz / norm, -1, 1)))
    phi = np.degrees(np.arctan2(My, Mx))
    inc = theta - 90.0
    dec = np.mod(90.0 - phi, 360.0)
    return dec, inc


def circular_diff_deg(a, b):
    """Shortest angular distance between two angles in degrees, range [0, 180]."""
    return abs((b - a + 180) % 360 - 180)


# ------------------------------------------------------------
# Emax bins
# ------------------------------------------------------------
emax_bins    = [0.0, 0.1, 0.3, 0.6, np.inf]
emax_labels  = [
    r"$E_{\max} < 0.1\ \mu m$",
    r"$E_{\max}\ 0.1$-$0.3\ \mu m$",
    r"$E_{\max}\ 0.3$-$0.6\ \mu m$",
    r"$E_{\max} > 0.6\ \mu m$",
]
emax_colors  = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728"]
emax_markers = ["o", "s", "^", "v"]

# ------------------------------------------------------------
# LOAD grain size
# ------------------------------------------------------------
grain_df = pd.read_csv(GRAIN_CSV)
grain_df["body_key"] = grain_df["body"].astype(str)
grain_df = grain_df.set_index("body_key")

# ------------------------------------------------------------
# LOAD R2 (Bz only)
# ------------------------------------------------------------
metrics_df = pd.read_csv(METRICS_CSV)
metrics_df = metrics_df[metrics_df["component"] == "Bz"].copy()
metrics_df = metrics_df.set_index("inversion")

# ------------------------------------------------------------
# LOAD body dirs
# ------------------------------------------------------------
body_dirs = sorted(
    d for d in os.listdir(FORWARD_ROOT)
    if d.startswith("body_") and os.path.isdir(os.path.join(FORWARD_ROOT, d))
)

# ------------------------------------------------------------
# LOOP over bodies and thresholds
# ------------------------------------------------------------
records = []

for body in body_dirs:
    # true moment - always from forward CSV (no threshold on forward)
    fwd_csv = glob.glob(os.path.join(FORWARD_ROOT, body, f"{body}_nonzero.csv"))
    if not fwd_csv:
        print(f"Skipping {body}: missing forward CSV.")
        continue

    fwd = pd.read_csv(fwd_csv[0], nrows=1)
    m_true = np.array([
        fwd["True_mx"].values[0],
        fwd["True_my"].values[0],
        fwd["True_mz"].values[0],
    ])
    norm_true = np.linalg.norm(m_true)
    if norm_true == 0:
        print(f"Skipping {body}: zero true moment.")
        continue
    dec_true, inc_true = vec_to_dec_inc(*m_true)

    # Emax
    grain_key = extract_grain_id(body)
    emax = grain_df.loc[grain_key, "Max_Elongation"] \
           if grain_key in grain_df.index else np.nan

    # R2
    r2 = metrics_df.loc[body, "r2"] if body in metrics_df.index else np.nan

    # loop over threshold CSVs
    for thresh in THRESHOLDS:
        thresh_str = f"{thresh:.1f}".replace(".", "p")
        inv_csv = glob.glob(os.path.join(
            INVERSE_ROOT, body,
            f"{body}_nonzero_thresh{thresh_str}.csv"
        ))

        if not inv_csv:
            print(f"  {body} thresh={thresh:.1f}: CSV not found - skipping.")
            continue

        inv = pd.read_csv(inv_csv[0], nrows=1)

        # check columns exist
        if "Recovered_mx" not in inv.columns:
            print(f"  {body} thresh={thresh:.1f}: no Recovered_mx column - skipping.")
            continue

        m_rec = np.array([
            inv["Recovered_mx"].values[0],
            inv["Recovered_my"].values[0],
            inv["Recovered_mz"].values[0],
        ])
        norm_rec = np.linalg.norm(m_rec)

        if norm_rec == 0:
            print(f"  {body} thresh={thresh:.1f}: zero recovered moment - skipping.")
            continue

        # Angular misfit (true 3D angle between the two moment vectors)
        cos_angle      = np.dot(m_true, m_rec) / (norm_true * norm_rec)
        angular_misfit = np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))

        # Amplitude misfit
        amplitude_misfit = abs(norm_rec - norm_true) / norm_true * 100

        # Dec/Inc of the recovered vector, and their individual angular differences
        dec_rec, inc_rec = vec_to_dec_inc(*m_rec)
        angular_dec = circular_diff_deg(dec_true, dec_rec)
        angular_inc = abs(inc_rec - inc_true)

        records.append({
            "body":             body,
            "threshold":        thresh,
            "Angular_misfit":   angular_misfit,
            "Amplitude_misfit": amplitude_misfit,
            "Angular_Dec":      angular_dec,
            "Angular_Inc":      angular_inc,
            "Dec_true":         dec_true,
            "Inc_true":         inc_true,
            "Dec_rec":          dec_rec,
            "Inc_rec":          inc_rec,
            "Emax":             emax,
            "R2":               r2,
        })

df_results = pd.DataFrame(records)
print(df_results.to_string())

#os.makedirs("output", exist_ok=True)
df_results.to_csv(os.path.join(INVERSE_ROOT, "misfit_vs_threshold.csv"), index=False)

# ------------------------------------------------------------
# PLOT - x axis = threshold, y axis = misfit (log),
#         one curve/scatter per grain colored by Emax bin
# ------------------------------------------------------------
TOLERANCE_ANG = 5.0
TOLERANCE_AMP = 5.0

PANELS = [
    ("Angular_misfit",   "Angular misfit (°)",     "(a) Angular misfit",     TOLERANCE_ANG),
    ("Amplitude_misfit", "Amplitude misfit (%)",   "(b) Amplitude misfit",   TOLERANCE_AMP),
    ("Angular_Dec",      "Declination misfit (°)", "(c) Declination misfit", TOLERANCE_ANG),
    ("Angular_Inc",      "Inclination misfit (°)", "(d) Inclination misfit", TOLERANCE_ANG),
]

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (column, ylabel, title, tolerance) in zip(axes, PANELS):
    # shaded tolerance band + dashed line
    ax.axhspan(0, tolerance, alpha=0.12, color="gray")
    ax.axhline(tolerance, color="gray", linestyle="--", linewidth=1.0)

    # plot each body as a line + scatter across thresholds
    for body, grp in df_results.groupby("body"):
        grp = grp.sort_values("threshold")

        emax = grp["Emax"].iloc[0]
        r2   = grp["R2"].iloc[0]

        if np.isnan(emax):
            col, mark = "gray", "o"
        else:
            b = emax_bin(emax)
            col, mark = emax_colors[b], emax_markers[b]

        size = r2_to_size(r2)

        ax.plot(grp["threshold"], grp[column], color=col, linewidth=0.8, alpha=0.6)
        ax.scatter(grp["threshold"], grp[column], color=col, marker=mark, s=size,
                   edgecolors="k", linewidths=0.4, zorder=3)

    ax.set_yscale("log")
    ax.set_xlabel("Threshold (fraction of max $|M|$)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xlim(-0.05, 1.0)
    ax.set_xticks(THRESHOLDS)
    ax.grid(True, which="both", linewidth=0.4, alpha=0.6)

# legend: Emax bins
legend_handles_emax = [
    plt.scatter([], [], color=emax_colors[i], marker=emax_markers[i],
                s=80, edgecolors="k", linewidths=0.4, label=emax_labels[i])
    for i in range(len(emax_labels))
]

# legend: R2 size tiers
legend_handles_r2 = [
    plt.scatter([], [], color="gray", marker="o", s=120,
                edgecolors="k", linewidths=0.4, label=r"$R^2 > 0.95$"),
    plt.scatter([], [], color="gray", marker="o", s=60,
                edgecolors="k", linewidths=0.4, label=r"$0.85 < R^2 \leq 0.95$"),
    plt.scatter([], [], color="gray", marker="o", s=20,
                edgecolors="k", linewidths=0.4, label=r"$R^2 \leq 0.85$"),
]

#axes[0].legend(
#    handles=legend_handles_emax + legend_handles_r2,
#    fontsize=9, loc="upper right", framealpha=0.9
#)

plt.tight_layout()
plt.savefig(os.path.join(INVERSE_ROOT, "misfit_per_threshold.png"), dpi=400, bbox_inches="tight")
plt.savefig(os.path.join(INVERSE_ROOT, "misfit_per_threshold.pdf"), bbox_inches="tight")
plt.show()

### 8_recovered misfits with heights

In [ ]:
#against height and size
%matplotlib inline

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.ticker import LogLocator, FuncFormatter

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
base_dir = os.getcwd()
FORWARD_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")
INVERSE_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_inversion")
METRICS_CSV  = os.path.join(INVERSE_ROOT, "inversion_metrics.csv")
GRAIN_CSV    = r"C:\Users\23121159\OneDrive - Curtin\Ongoing\JGR_Review\Work\1-Merrill\Particle_morphology_data\Grain_size_distirbution.csv"

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      13,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "pdf.fonttype":   42,
    "ps.fonttype":    42,
})

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def extract_grain_id(body_name):
    """body_118_24_mult_h0.010 -> '118' (unaffected by the trailing height tag)"""
    return body_name.split("_")[1]


def extract_height_um(body_name):
    """
    body_118_24_mult_h0.010 -> 0.010 (float, same units SENSOR_HEIGHT used,
    µm). Reads the trailing '_h<number>' tag added by tec_converter.py's
    height-sweep naming convention. Returns np.nan if no tag is found
    (e.g. an older body folder generated before the height sweep existed).
    """
    m = re.search(r"_h([0-9]*\.?[0-9]+)$", body_name)
    if m is None:
        return np.nan
    return float(m.group(1))


def r2_to_size(r2):
    if np.isnan(r2):
        return 60
    if r2 > 0.95:
        return 120
    elif r2 > 0.85:
        return 60
    else:
        return 20


def vec_to_dec_inc(Mx, My, Mz):
    """
    Vector -> (Dec, Inc) in degrees, using the same convention as the rest
    of the pipeline (Mz flipped internally, Inc = theta-90, Dec = 90-phi).
    """
    norm = np.sqrt(Mx ** 2 + My ** 2 + Mz ** 2)
    theta = np.degrees(np.arccos(np.clip(-Mz / norm, -1, 1)))
    phi = np.degrees(np.arctan2(My, Mx))
    inc = theta - 90.0
    dec = np.mod(90.0 - phi, 360.0)
    return dec, inc


def circular_diff_deg(a, b):
    """Shortest angular distance between two angles in degrees, range [0, 180]."""
    return abs((b - a + 180) % 360 - 180)


def decimal_formatter(val, pos):
    """Format log-scale tick labels as plain decimals (0.01, 0.001, ...)
    instead of scientific notation (1e-2, 1e-3, ...)."""
    if val == 0:
        return "0"
    s = f"{val:.10f}".rstrip("0").rstrip(".")
    return s


# ------------------------------------------------------------
# LOAD grain size (Emax)
# body column in CSV is just the number: 9, 12, 14 ...
# ------------------------------------------------------------
grain_df = pd.read_csv(GRAIN_CSV)
grain_df["body_key"] = grain_df["body"].astype(str)
grain_df = grain_df.set_index("body_key")

# ------------------------------------------------------------
# LOAD R2 from metrics CSV (Bz component only)
# ------------------------------------------------------------
metrics_df = pd.read_csv(METRICS_CSV)
metrics_df = metrics_df[metrics_df["component"] == "Bz"].copy()
metrics_df = metrics_df.set_index("inversion")

# ------------------------------------------------------------
# LOAD true and recovered moments, compute misfits
# ------------------------------------------------------------
body_dirs = sorted(
    d for d in os.listdir(FORWARD_ROOT)
    if d.startswith("body_") and os.path.isdir(os.path.join(FORWARD_ROOT, d))
)

records = []

for body in body_dirs:
    fwd_csv = glob.glob(os.path.join(FORWARD_ROOT, body, f"{body}_nonzero.csv"))
    inv_csv = glob.glob(os.path.join(INVERSE_ROOT, body, f"{body}_nonzero.csv"))

    if not fwd_csv or not inv_csv:
        print(f"Skipping {body}: missing CSV.")
        continue

    sensor_height = extract_height_um(body)
    if np.isnan(sensor_height):
        print(f"Skipping {body}: could not parse a sensor height from the body name "
              f"(expected a trailing '_h<value>' tag, e.g. '..._h0.010').")
        continue

    fwd = pd.read_csv(fwd_csv[0], nrows=1)
    inv = pd.read_csv(inv_csv[0], nrows=1)

    m_true = np.array([
        fwd["True_mx"].values[0],
        fwd["True_my"].values[0],
        fwd["True_mz"].values[0],
    ])
    m_rec = np.array([
        inv["Recovered_mx"].values[0],
        inv["Recovered_my"].values[0],
        inv["Recovered_mz"].values[0],
    ])

    norm_true = np.linalg.norm(m_true)
    norm_rec  = np.linalg.norm(m_rec)

    if norm_true == 0 or norm_rec == 0:
        print(f"Skipping {body}: zero moment.")
        continue

    # Angular misfit (true 3D angle between the two moment vectors)
    cos_angle      = np.dot(m_true, m_rec) / (norm_true * norm_rec)
    angular_misfit = np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))

    # Amplitude misfit (%)
    amplitude_misfit = abs(norm_rec - norm_true) / norm_true * 100

    # Dec/Inc of each vector, and their individual angular differences
    dec_true, inc_true = vec_to_dec_inc(*m_true)
    dec_rec, inc_rec   = vec_to_dec_inc(*m_rec)
    angular_dec = circular_diff_deg(dec_true, dec_rec)   # wraps correctly at 0/360
    angular_inc = abs(inc_rec - inc_true)                # inc has no wraparound (-90..90)

    # Emax lookup: extract grain number from body name
    grain_key = extract_grain_id(body)
    emax = grain_df.loc[grain_key, "Max_Elongation"] if grain_key in grain_df.index else np.nan
    if np.isnan(emax):
        print(f"  Warning: no Emax found for {body} (grain_key={grain_key})")

    # R2 lookup
    r2 = metrics_df.loc[body, "r2"] if body in metrics_df.index else np.nan

    records.append({
        "body":             body,
        "grain_key":        grain_key,
        "sensor_height":    sensor_height,
        "Angular_misfit":   angular_misfit,
        "Amplitude_misfit": amplitude_misfit,
        "Angular_Dec":      angular_dec,
        "Angular_Inc":      angular_inc,
        "Dec_true":         dec_true,
        "Inc_true":         inc_true,
        "Dec_rec":          dec_rec,
        "Inc_rec":          inc_rec,
        "Emax":             emax,
        "R2":               r2,
    })

df_results = pd.DataFrame(records)
print(df_results.to_string())

#os.makedirs("output", exist_ok=True)
df_results.to_csv(os.path.join(INVERSE_ROOT, "misfit_results.csv"), index=False)

# ------------------------------------------------------------
# Build color/marker map from the EXACT distinct Emax values present
# in the data (not fixed bins/ranges) -- each unique Emax value gets
# its own color and filled marker shape.
# ------------------------------------------------------------
unique_emax = np.sort(df_results["Emax"].dropna().unique())

_marker_cycle = ["o", "s", "^", "v", "D", "P", "X", "*", "<", ">", "h", "8"]
_color_cycle  = plt.cm.tab10.colors  # 10 distinct colors, cycles if more values than that

emax_color_map  = {val: _color_cycle[i % len(_color_cycle)] for i, val in enumerate(unique_emax)}
emax_marker_map = {val: _marker_cycle[i % len(_marker_cycle)] for i, val in enumerate(unique_emax)}

# ------------------------------------------------------------
# PLOT -- styled to match the published figure: log-log axes,
# fixed y-range with explicit decade ticks shown as plain decimals,
# full flat shading below tolerance (not just a thin band), dashed
# lines at the actual swept heights present in the data, filled
# markers, and legend entries keyed to each exact Emax value.
# ------------------------------------------------------------
TOLERANCE_ANG = 5.0
TOLERANCE_AMP = 10.0   # matches the paper figure's shading boundary

def auto_log_ylim_ticks(series, tolerance, pad_decades=0.5):
    """Compute y-limits/ticks spanning just the actual data range (plus the
    tolerance line and a little padding), instead of a fixed wide range."""
    vals = series.replace(0, np.nan).dropna()
    vals = vals[vals > 0]
    if len(vals) == 0:
        lo, hi = tolerance / 10, tolerance * 10
    else:
        lo = min(vals.min(), tolerance)
        hi = max(vals.max(), tolerance)
    log_lo = np.floor(np.log10(lo) - pad_decades)
    log_hi = np.ceil(np.log10(hi) + pad_decades)
    ylim = (10 ** log_lo, 10 ** log_hi)
    yticks = [10 ** e for e in range(int(log_hi), int(log_lo) - 1, -1)]
    return ylim, yticks

YLIM_ANG, YTICKS_ANG = auto_log_ylim_ticks(df_results["Angular_misfit"], TOLERANCE_ANG)
YLIM_AMP, YTICKS_AMP = auto_log_ylim_ticks(df_results["Amplitude_misfit"], TOLERANCE_AMP)

PANELS = [
    ("Angular_misfit",   "Angular misfit (°)",     "Full Model Inversion",      TOLERANCE_ANG, YLIM_ANG, YTICKS_ANG),
    ("Amplitude_misfit", "Amplitude misfit (%)",   "Full Model Inversion",    TOLERANCE_AMP, YLIM_AMP, YTICKS_AMP),
    ("Angular_Dec",      "Declination misfit (°)", "Full Model Inversion", TOLERANCE_ANG, YLIM_ANG, YTICKS_ANG),
    ("Angular_Inc",      "Inclination misfit (°)", "Full Model Inversion", TOLERANCE_ANG, YLIM_ANG, YTICKS_ANG),
]

# reference heights actually swept (not hardcoded) -- drawn as dashed
# vertical lines with rotated labels, same as the paper figure
all_heights = np.sort(df_results["sensor_height"].dropna().unique())
xmin = all_heights.min() / 1.5 if len(all_heights) else 0.005
xmax = all_heights.max() * 1.5 if len(all_heights) else 50

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (column, ylabel, title, tolerance, ylim, yticks) in zip(axes, PANELS):
    # full flat shaded "acceptable fit" region below tolerance
    ax.axhspan(ylim[0], tolerance, color="#e6e6e6", zorder=0)

    # connect points from the same grain across heights with a line,
    # so you can trace how each grain's misfit moves as height changes
    for grain_key, grp in df_results.groupby("grain_key"):
        grp = grp.sort_values("sensor_height")
        if len(grp) < 2:
            continue
        emax_val = grp["Emax"].iloc[0]
        line_col = emax_color_map.get(emax_val, "gray")
        ax.plot(grp["sensor_height"], grp[column], color=line_col,
                linewidth=1.0, alpha=0.6, zorder=2)

    for _, row in df_results.iterrows():
        emax = row["Emax"]
        r2   = row["R2"]

        if np.isnan(emax):
            col, mark = "gray", "o"
        else:
            col, mark = emax_color_map[emax], emax_marker_map[emax]

        size = r2_to_size(r2)
        ax.scatter(row["sensor_height"], row[column], facecolor=col, edgecolors=col,
                   marker=mark, s=size, linewidths=1.0, zorder=3)

    # dashed reference height lines + rotated labels (actual sweep, not hardcoded)
    for h in all_heights:
        ax.axvline(h, color="black", linestyle="--", linewidth=2.2, zorder=2)
        ax.text(h, ylim[1] * 0.55, f"{h:.2f} \u00b5m", rotation=90,
                va="top", ha="center", fontsize=8)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(*ylim)
    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=20))
    ax.set_yticks(yticks)
    ax.yaxis.set_major_formatter(FuncFormatter(decimal_formatter))
    ax.xaxis.set_major_formatter(FuncFormatter(decimal_formatter))
    ax.set_xlabel("Source to Sensor distance (µm)")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(False)

# legend: exact Emax values, each with its own filled color/marker
legend_handles_emax = [
    plt.scatter([], [], facecolor=emax_color_map[val], edgecolor=emax_color_map[val], marker=emax_marker_map[val],
                s=80, linewidths=1.0, label=fr"$E_{{\max}} = {val:.2f}\ \mu m$")
    for val in unique_emax
]

# legend: R2 size tiers
legend_handles_r2 = [
    plt.scatter([], [], facecolor="none", edgecolor="black", marker="o", s=120,
                linewidths=1.0, label=r"$R^2 > 0.95$"),
    plt.scatter([], [], facecolor="none", edgecolor="black", marker="o", s=60,
                linewidths=1.0, label=r"$0.85 < R^2 \leq 0.95$"),
    plt.scatter([], [], facecolor="none", edgecolor="black", marker="o", s=20,
                linewidths=1.0, label=r"$R^2 \leq 0.85$"),
]

axes[0].legend(
    handles=legend_handles_emax + legend_handles_r2,
    fontsize=9, loc="lower left", framealpha=0.95
)

plt.tight_layout()
plt.savefig(os.path.join(INVERSE_ROOT, "misfit_per_grain.png"), dpi=400, bbox_inches="tight")
plt.savefig(os.path.join(INVERSE_ROOT, "misfit_per_grain.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
#against grain size (Emax), fixed height = 0.01 um
%matplotlib inline

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import cm
from matplotlib.ticker import LogLocator, FuncFormatter

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
base_dir = os.getcwd()
FORWARD_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")
INVERSE_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_inversion")
METRICS_CSV  = os.path.join(INVERSE_ROOT, "inversion_metrics.csv")
GRAIN_CSV    = r"C:\Users\23121159\OneDrive - Curtin\Ongoing\JGR_Review\Work\1-Merrill\Particle_morphology_data\Grain_size_distirbution.csv"

TARGET_HEIGHT_UM = 0.01   # only plot this sensor height

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      13,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "pdf.fonttype":   42,
    "ps.fonttype":    42,
})

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------

def emax_to_domain_state(emax):
    """Rough domain-state label from grain size, per the stated assumption:
    SD <= 0.1 um, SV 0.1-0.6 um, MV > 0.6 um."""
    if emax <= 0.1:
        return "SD"
    elif emax <= 0.61:
        return "SV"
    else:
        return "MV"
        
def extract_grain_id(body_name):
    """body_118_24_mult_h0.010 -> '118' (unaffected by the trailing height tag)"""
    return body_name.split("_")[1]


def extract_height_um(body_name):
    """
    body_118_24_mult_h0.010 -> 0.010 (float, same units SENSOR_HEIGHT used,
    µm). Reads the trailing '_h<number>' tag added by tec_converter.py's
    height-sweep naming convention. Returns np.nan if no tag is found
    (e.g. an older body folder generated before the height sweep existed).
    """
    m = re.search(r"_h([0-9]*\.?[0-9]+)$", body_name)
    if m is None:
        return np.nan
    return float(m.group(1))


def r2_to_size(r2):
    if np.isnan(r2):
        return 60
    if r2 > 0.95:
        return 120
    elif r2 > 0.85:
        return 60
    else:
        return 20


def vec_to_dec_inc(Mx, My, Mz):
    """
    Vector -> (Dec, Inc) in degrees, using the same convention as the rest
    of the pipeline (Mz flipped internally, Inc = theta-90, Dec = 90-phi).
    """
    norm = np.sqrt(Mx ** 2 + My ** 2 + Mz ** 2)
    theta = np.degrees(np.arccos(np.clip(-Mz / norm, -1, 1)))
    phi = np.degrees(np.arctan2(My, Mx))
    inc = theta - 90.0
    dec = np.mod(90.0 - phi, 360.0)
    return dec, inc


def circular_diff_deg(a, b):
    """Shortest angular distance between two angles in degrees, range [0, 180]."""
    return abs((b - a + 180) % 360 - 180)


def decimal_formatter(val, pos):
    """Format log-scale tick labels as plain decimals (0.01, 0.001, ...)
    instead of scientific notation (1e-2, 1e-3, ...)."""
    if val == 0:
        return "0"
    s = f"{val:.10f}".rstrip("0").rstrip(".")
    return s


# ------------------------------------------------------------
# LOAD grain size (Emax)
# body column in CSV is just the number: 9, 12, 14 ...
# ------------------------------------------------------------
grain_df = pd.read_csv(GRAIN_CSV)
grain_df["body_key"] = grain_df["body"].astype(str)
grain_df = grain_df.set_index("body_key")

# ------------------------------------------------------------
# LOAD R2 from metrics CSV (Bz component only)
# ------------------------------------------------------------
metrics_df = pd.read_csv(METRICS_CSV)
metrics_df = metrics_df[metrics_df["component"] == "Bz"].copy()
metrics_df = metrics_df.set_index("inversion")

# ------------------------------------------------------------
# LOAD true and recovered moments, compute misfits
# ------------------------------------------------------------
body_dirs = sorted(
    d for d in os.listdir(FORWARD_ROOT)
    if d.startswith("body_") and os.path.isdir(os.path.join(FORWARD_ROOT, d))
)

records = []

for body in body_dirs:
    fwd_csv = glob.glob(os.path.join(FORWARD_ROOT, body, f"{body}_nonzero.csv"))
    inv_csv = glob.glob(os.path.join(INVERSE_ROOT, body, f"{body}_nonzero.csv"))

    if not fwd_csv or not inv_csv:
        print(f"Skipping {body}: missing CSV.")
        continue

    sensor_height = extract_height_um(body)
    if np.isnan(sensor_height):
        print(f"Skipping {body}: could not parse a sensor height from the body name "
              f"(expected a trailing '_h<value>' tag, e.g. '..._h0.010').")
        continue

    # Only keep the target height
    if not np.isclose(sensor_height, TARGET_HEIGHT_UM, atol=1e-6):
        continue

    fwd = pd.read_csv(fwd_csv[0], nrows=1)
    inv = pd.read_csv(inv_csv[0], nrows=1)

    m_true = np.array([
        fwd["True_mx"].values[0],
        fwd["True_my"].values[0],
        fwd["True_mz"].values[0],
    ])
    m_rec = np.array([
        inv["Recovered_mx"].values[0],
        inv["Recovered_my"].values[0],
        inv["Recovered_mz"].values[0],
    ])

    norm_true = np.linalg.norm(m_true)
    norm_rec  = np.linalg.norm(m_rec)

    if norm_true == 0 or norm_rec == 0:
        print(f"Skipping {body}: zero moment.")
        continue

    # Angular misfit (true 3D angle between the two moment vectors)
    cos_angle      = np.dot(m_true, m_rec) / (norm_true * norm_rec)
    angular_misfit = np.degrees(np.arccos(np.clip(cos_angle, -1, 1)))

    # Amplitude misfit (%)
    amplitude_misfit = abs(norm_rec - norm_true) / norm_true * 100

    # Dec/Inc of each vector, and their individual angular differences
    dec_true, inc_true = vec_to_dec_inc(*m_true)
    dec_rec, inc_rec   = vec_to_dec_inc(*m_rec)
    angular_dec = circular_diff_deg(dec_true, dec_rec)   # wraps correctly at 0/360
    angular_inc = abs(inc_rec - inc_true)                # inc has no wraparound (-90..90)

    # Emax lookup: extract grain number from body name
    grain_key = extract_grain_id(body)
    emax = grain_df.loc[grain_key, "Max_Elongation"] if grain_key in grain_df.index else np.nan
    if np.isnan(emax):
        print(f"  Warning: no Emax found for {body} (grain_key={grain_key})")

    # R2 lookup
    r2 = metrics_df.loc[body, "r2"] if body in metrics_df.index else np.nan

    records.append({
        "body":             body,
        "grain_key":        grain_key,
        "sensor_height":    sensor_height,
        "Angular_misfit":   angular_misfit,
        "Amplitude_misfit": amplitude_misfit,
        "Angular_Dec":      angular_dec,
        "Angular_Inc":      angular_inc,
        "Dec_true":         dec_true,
        "Inc_true":         inc_true,
        "Dec_rec":          dec_rec,
        "Inc_rec":          inc_rec,
        "Emax":             emax,
        "R2":               r2,
    })

df_results = pd.DataFrame(records)
print(df_results.to_string())

df_results.to_csv(os.path.join(INVERSE_ROOT, f"misfit_results_h{TARGET_HEIGHT_UM:.3f}.csv"), index=False)

# ------------------------------------------------------------
# Build color/marker map from the EXACT distinct Emax values present
# in the data (not fixed bins/ranges) -- each unique Emax value gets
# its own color and filled marker shape.
# ------------------------------------------------------------
unique_emax = np.sort(df_results["Emax"].dropna().unique())

_color_cycle  = plt.cm.tab10.colors  # 10 distinct colors, cycles if more values than that

emax_color_map  = {val: _color_cycle[i % len(_color_cycle)] for i, val in enumerate(unique_emax)}
emax_marker_map = {val: "o" for val in unique_emax}   # same marker (filled circle) for every Emax value

# ------------------------------------------------------------
# PLOT -- x-axis is Emax (grain size), data restricted to
# TARGET_HEIGHT_UM. No shading, legend in top-left of first panel.
# ------------------------------------------------------------
TOLERANCE_ANG = 5.0
TOLERANCE_AMP = 10.0

def auto_log_ylim_ticks(series, tolerance, pad_decades=0.5):
    """Compute y-limits/ticks spanning just the actual data range (plus the
    tolerance line and a little padding), instead of a fixed wide range."""
    vals = series.replace(0, np.nan).dropna()
    vals = vals[vals > 0]
    if len(vals) == 0:
        lo, hi = tolerance / 10, tolerance * 10
    else:
        lo = min(vals.min(), tolerance)
        hi = max(vals.max(), tolerance)
    log_lo = np.floor(np.log10(lo) - pad_decades)
    log_hi = np.ceil(np.log10(hi) + pad_decades)
    ylim = (10 ** log_lo, 10 ** log_hi)
    yticks = [10 ** e for e in range(int(log_hi), int(log_lo) - 1, -1)]
    return ylim, yticks

YLIM_ANG, YTICKS_ANG = auto_log_ylim_ticks(df_results["Angular_misfit"], TOLERANCE_ANG)
YLIM_AMP, YTICKS_AMP = auto_log_ylim_ticks(df_results["Amplitude_misfit"], TOLERANCE_AMP)

PANELS = [
    ("Angular_misfit",   "Angular misfit (°)",     "Full Model Inversion",      TOLERANCE_ANG, YLIM_ANG, YTICKS_ANG),
    ("Amplitude_misfit", "Amplitude misfit (%)",   "Full Model Inversion",    TOLERANCE_AMP, YLIM_AMP, YTICKS_AMP),
    ("Angular_Dec",      "Declination misfit (°)", "Full Model Inversion", TOLERANCE_ANG, YLIM_ANG, YTICKS_ANG),
    ("Angular_Inc",      "Inclination misfit (°)", "Full Model Inversion", TOLERANCE_ANG, YLIM_ANG, YTICKS_ANG),
]

# x-range spans the actual Emax values present (log-spaced)
all_emax = np.sort(df_results["Emax"].dropna().unique())
xmin = all_emax.min() / 1.5 if len(all_emax) else 0.01
xmax = all_emax.max() * 1.5 if len(all_emax) else 2.0

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for ax, (column, ylabel, title, tolerance, ylim, yticks) in zip(axes, PANELS):
    for _, row in df_results.iterrows():
        emax = row["Emax"]
        r2   = row["R2"]

        if np.isnan(emax):
            col, mark = "gray", "o"
        else:
            col, mark = emax_color_map[emax], emax_marker_map[emax]

        size = r2_to_size(r2)
        ax.scatter(row["Emax"], row[column], facecolor=col, edgecolors=col,
                   marker=mark, s=size, linewidths=1.0, zorder=3)

    ax.set_xscale("log")
    ax.set_yscale("log")
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(*ylim)
    ax.yaxis.set_major_locator(LogLocator(base=10.0, numticks=20))
    ax.set_yticks(yticks)
    ax.yaxis.set_major_formatter(FuncFormatter(decimal_formatter))
    ax.xaxis.set_major_formatter(FuncFormatter(decimal_formatter))
    ax.set_xlabel(r"Grain size, $E_{\max}$ (µm)")
    ax.set_ylabel(ylabel)
    ax.set_title(f"{title}  (h = {TARGET_HEIGHT_UM:.3f} µm)")
    ax.grid(False)

# legend: exact Emax values, each with its own filled color/marker
legend_handles_emax = [
    plt.scatter([], [], facecolor=emax_color_map[val], edgecolor=emax_color_map[val], marker=emax_marker_map[val],
                s=80, linewidths=1.0,
                label=fr"{emax_to_domain_state(val)}, $E_{{\max}} = {val:.2f}\ \mu m$")
    for val in unique_emax
]

# legend: R2 size tiers
legend_handles_r2 = [
    plt.scatter([], [], facecolor="none", edgecolor="black", marker="o", s=120,
                linewidths=1.0, label=r"$R^2 > 0.95$"),
    plt.scatter([], [], facecolor="none", edgecolor="black", marker="o", s=60,
                linewidths=1.0, label=r"$0.85 < R^2 \leq 0.95$"),
    plt.scatter([], [], facecolor="none", edgecolor="black", marker="o", s=20,
                linewidths=1.0, label=r"$R^2 \leq 0.85$"),
]

axes[0].legend(
    handles=legend_handles_emax + legend_handles_r2,
    fontsize=9, loc="upper left", framealpha=0.95
)

plt.tight_layout()
plt.savefig(os.path.join(INVERSE_ROOT, f"misfit_per_grain_h{TARGET_HEIGHT_UM:.3f}.png"), dpi=400, bbox_inches="tight")
plt.savefig(os.path.join(INVERSE_ROOT, f"misfit_per_grain_h{TARGET_HEIGHT_UM:.3f}.pdf"), bbox_inches="tight")
plt.show()

In [ ]:
# R2 vs sensor height - INVERSION version
# Plots R²_dip (Bz component) against sensor height extracted from body
# folder names, colour/marker-coded by EXACT Emax value (not bins), with
# each grain's points connected by a line across heights so you can trace
# how R² moves as height changes.
%matplotlib inline

import os
import re
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------
base_dir     = os.getcwd()
FORWARD_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "tomofast_forward")
INVERSE_ROOT = os.path.join(base_dir, "Tec_files_bellon", "converted", "Tomofast_Output_inversion")
METRICS_CSV  = os.path.join(INVERSE_ROOT, "inversion_metrics.csv")
GRAIN_CSV    = r"C:\Users\23121159\OneDrive - Curtin\Ongoing\JGR_Review\Work\1-Merrill\Particle_morphology_data\Grain_size_distirbution.csv"

plt.rcParams.update({
    "font.family":    "serif",
    "font.size":      13,
    "axes.labelsize": 13,
    "axes.titlesize": 14,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "pdf.fonttype":   42,
    "ps.fonttype":    42,
})

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def extract_grain_id(body_name):
    """body_118_24_mult_h0.010 -> '118' (unaffected by the trailing height tag)"""
    return body_name.split("_")[1]


def extract_height_um(body_name):
    m = re.search(r"_h([0-9]*\.?[0-9]+)$", body_name)
    return float(m.group(1)) if m else np.nan


def decimal_formatter(val, pos):
    """Format log-scale tick labels as plain decimals (0.01, 0.001, ...)
    instead of scientific notation (1e-2, 1e-3, ...)."""
    if val == 0:
        return "0"
    s = f"{val:.10f}".rstrip("0").rstrip(".")
    return s


# ------------------------------------------------------------
# LOAD grain size (Emax)
# ------------------------------------------------------------
grain_df = pd.read_csv(GRAIN_CSV)
grain_df["body_key"] = grain_df["body"].astype(str)
grain_df = grain_df.set_index("body_key")

# ------------------------------------------------------------
# LOAD R² from metrics CSV (Bz component only)
# ------------------------------------------------------------
metrics_df = pd.read_csv(METRICS_CSV)
metrics_df = metrics_df[metrics_df["component"] == "Bz"].copy()
metrics_df = metrics_df.set_index("inversion")

# ------------------------------------------------------------
# BUILD records: one row per (body, height) with its R² and Emax
# ------------------------------------------------------------
body_dirs = sorted(
    d for d in os.listdir(FORWARD_ROOT)
    if d.startswith("body_") and os.path.isdir(os.path.join(FORWARD_ROOT, d))
)

records = []
for body in body_dirs:
    sensor_height = extract_height_um(body)
    if np.isnan(sensor_height):
        print(f"Skipping {body}: no height tag.")
        continue

    r2 = metrics_df.loc[body, "r2"] if body in metrics_df.index else np.nan

    grain_key = extract_grain_id(body)
    emax = grain_df.loc[grain_key, "Max_Elongation"] if grain_key in grain_df.index else np.nan

    records.append({
        "body":          body,
        "grain_key":     grain_key,
        "sensor_height": sensor_height,
        "R2":            r2,
        "Emax":          emax,
    })

df = pd.DataFrame(records)
print(df.to_string())
df.to_csv(os.path.join(INVERSE_ROOT, "r2_vs_height_inversion.csv"), index=False)

# ------------------------------------------------------------
# Build color/marker map from the EXACT distinct Emax values present
# in the data (not fixed bins/ranges) -- each unique Emax value gets
# its own color and filled marker shape, matching the misfit plots.
# ------------------------------------------------------------
unique_emax = np.sort(df["Emax"].dropna().unique())

_marker_cycle = ["o", "s", "^", "v", "D", "P", "X", "*", "<", ">", "h", "8"]
_color_cycle  = plt.cm.tab10.colors  # 10 distinct colors, cycles if more values than that

emax_color_map  = {val: _color_cycle[i % len(_color_cycle)] for i, val in enumerate(unique_emax)}
emax_marker_map = {val: _marker_cycle[i % len(_marker_cycle)] for i, val in enumerate(unique_emax)}

heights_sorted = sorted(df["sensor_height"].unique())

# ------------------------------------------------------------
# PLOT
# ------------------------------------------------------------
fig, ax = plt.subplots(figsize=(12, 5))

# Reference lines
ax.axhline(0.90, color="black", linestyle="--", linewidth=1.0, zorder=1)
ax.axhline(0.50, color="black", linestyle="--", linewidth=1.0, zorder=1)

# connect points from the same grain across heights with a line,
# so you can trace how each grain's R2 moves as height changes
for grain_key, grp in df.groupby("grain_key"):
    grp = grp.dropna(subset=["R2"]).sort_values("sensor_height")
    if len(grp) < 2:
        continue
    emax_val = grp["Emax"].iloc[0]
    line_col = emax_color_map.get(emax_val, "gray")
    ax.plot(grp["sensor_height"], grp["R2"], color=line_col,
            linewidth=1.0, alpha=0.6, zorder=2)

for _, row in df.iterrows():
    if np.isnan(row["R2"]):
        continue
    emax = row["Emax"]
    if np.isnan(emax):
        col, mark = "gray", "o"
    else:
        col, mark = emax_color_map[emax], emax_marker_map[emax]

    ax.scatter(row["sensor_height"], row["R2"], facecolor=col, edgecolors=col,
               marker=mark, s=60, linewidths=1.0, zorder=3)

ax.set_xscale("log")
ax.xaxis.set_major_formatter(FuncFormatter(decimal_formatter))
ax.set_xlabel("Source to Sensor distance (µm)")
ax.set_ylabel(r"$R^2_{\mathrm{Inversion}}$")
ax.set_title("(a) Inversion - $R^2$ vs sensor height")
ax.set_xlim(left=min(heights_sorted) / 1.5, right=max(heights_sorted) * 1.5)
ax.set_ylim(-1.0, 1.05)
ax.grid(True, axis="y", linewidth=0.4, alpha=0.6)
ax.grid(False, axis="x")

# legend: exact Emax values, each with its own filled color/marker
legend_handles = [
    plt.scatter([], [], facecolor=emax_color_map[val], edgecolor=emax_color_map[val],
                marker=emax_marker_map[val], s=80, linewidths=1.0,
                label=fr"$E_{{\max}} = {val:.2f}\ \mu m$")
    for val in unique_emax
]
ax.legend(handles=legend_handles, title=r"$E_{\max}$", fontsize=10,
          title_fontsize=10, loc="lower right", framealpha=0.9)

plt.tight_layout()
plt.savefig(os.path.join(INVERSE_ROOT, "r2_vs_height_inversion.png"), dpi=400, bbox_inches="tight")
plt.savefig(os.path.join(INVERSE_ROOT, "r2_vs_height_inversion.pdf"), bbox_inches="tight")
plt.show()